In [1]:
from collections import OrderedDict

from qiskit_metal import designs, Dict
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.tlines.anchored_path import RouteAnchors
from qiskit_metal.qlibrary.tlines.pathfinder import RoutePathfinder
from qiskit_metal.qlibrary.terminations.launchpad_wb import LaunchpadWirebond
from qiskit_metal.qlibrary.terminations.short_to_ground import ShortToGround
from qiskit_metal.qlibrary.couplers.coupled_line_tee import CoupledLineTee
from qiskit_metal.qlibrary.qubits.circle_transmon import CircTransmon
from qiskit_metal.qlibrary.qubits.squid_loop_jj import SquidLoopJJ
from qiskit_metal.qlibrary.qubits.squid_array import SquidArray
from qiskit_metal.qlibrary.sample_shapes.rect_cutout import RectCutout

from qiskit_metal import MetalGUI

import numpy as np

In [2]:
design = designs.DesignPlanar({}, overwrite_enabled=True)
design.chips.main.size.size_x = '10.0mm'
design.chips.main.size.size_y = '4.0mm'
design.chips.main.size.size_z = '500um' # tamaño del sustrato y de la capa de aire a simular
design.chips.main.size.center_x = '0mm'
design.chips.main.size.center_y = '0.0mm'

design.variables['cpw_width'] = '20 um'
design.variables['cpw_gap'] = '11.5um'  # ~50 Ohm at cpw_width=20um, see the DesignHelper check below

In [3]:
gui = MetalGUI(design)
gui.rebuild()
gui.autoscale()

In [4]:
try:
    LP1.delete(); LP2.delete(); CLT.delete(); TL1.delete(); TL2.delete()
except NameError:
    pass

fillet_radius_um = 75

def build_feedline_and_coupler(coupling_space_um, coupling_length_um, down_length_um=150.0,
                               chip_half_span_um=5000.0):
    """Feedline (LP1 -- TL1 -- CoupledLineTee -- TL2 -- LP2), with the
    resonator's coupled ("hanger") end built into the CoupledLineTee itself.

    ``coupling_space_um`` is the separation to the transmission line and
    ``coupling_length_um`` is the colinear coupled length -- the two
    parameters to sweep for the coupling quality factor Qc.
    ``down_length_um`` is normally DERIVED from a fixed budget (see below), not
    passed independently -- see build_resonator's docstring for why.
    """
    assert coupling_length_um < 0.8 * chip_half_span_um, (
        f"coupling_length_um={coupling_length_um} too large for the "
        f"{chip_half_span_um}um launchpad-to-center span")
    assert down_length_um > 0, (
        f"down_length_um={down_length_um} <= 0 (coupling_length_um={coupling_length_um} "
        f"exceeds COUPLING_BUDGET_UM) -- CoupledLineTee's own 'down' segment then runs "
        f"BACKWARDS (toward the feedline instead of away from it), which is exactly what "
        f"produced the resonator-overlaps-the-TL bug once. Raise COUPLING_BUDGET_UM.")

    x1, y1 = f'-{chip_half_span_um}um', '0um'
    x2 = f'{chip_half_span_um}um'

    launch_options1 = dict(chip='main', pos_x=x1, pos_y=y1, orientation='360',
                           lead_length='30um', pad_height='400um', pad_width='250um',
                           pad_gap='150um', taper_height='350um')
    lp1 = LaunchpadWirebond(design, 'LP1', options=launch_options1)

    launch_options2 = dict(chip='main', pos_x=x2, pos_y=y1, orientation='180',
                           lead_length='30um', pad_height='400um', pad_width='250um',
                           pad_gap='150um', taper_height='350um')
    lp2 = LaunchpadWirebond(design, 'LP2', options=launch_options2)

    # prime_* / second_* match the main cpw_width/cpw_gap -- the resonator's own
    # cross-section is the same 50-Ohm-ish line verified above, not a separate one.
    clt = CoupledLineTee(design, 'CLT', options=dict(
        pos_x='0um', pos_y='0um', orientation='0', chip='main',
        prime_width='cpw_width', prime_gap='cpw_gap',
        second_width='cpw_width', second_gap='cpw_gap',
        coupling_space=f'{coupling_space_um}um', coupling_length=f'{coupling_length_um}um',
        down_length=f'{down_length_um}um', fillet=f'{fillet_radius_um}um', mirror=False,
        open_termination=True))

    tl1 = RoutePathfinder(design, 'TL1', options=dict(
        chip='main', trace_width='cpw_width', trace_gap='cpw_gap', fillet=f'{fillet_radius_um}um',
        hfss_wire_bonds=True, lead=dict(end_straight='0.1mm'),
        pin_inputs=Dict(start_pin=Dict(component='LP1', pin='tie'),
                        end_pin=Dict(component='CLT', pin='prime_start'))))
    tl2 = RoutePathfinder(design, 'TL2', options=dict(
        chip='main', trace_width='cpw_width', trace_gap='cpw_gap', fillet=f'{fillet_radius_um}um',
        hfss_wire_bonds=True, lead=dict(start_straight='0.1mm'),
        pin_inputs=Dict(start_pin=Dict(component='CLT', pin='prime_end'),
                        end_pin=Dict(component='LP2', pin='tie'))))
    return lp1, lp2, clt, tl1, tl2


# --- sweep parameters: separation to the TL and colinear coupled length ---
COUPLING_SPACE_UM = 15.0     # "separación a la línea de transmisión"
COUPLING_LENGTH_UM = 200.0   # "longitud de acoplamiento" (colinear length)

# coupling_length_um + down_length_um is held at this fixed budget, so
# down_length_um below is DERIVED, not chosen independently -- this keeps
# res_short's position RELATIVE to CLT's second_end pin identical no matter
# what coupling_length_um/coupling_space_um are (see build_resonator's
# docstring), which keeps the meander's shape identical across the whole
# sweep too: only the coupled region changes, not the resonator's overall
# geometry -- nothing "drags" the meander body around between sweep points.
#
# MUST stay comfortably above the largest coupling_length_um you plan to
# sweep (leaving down_length_um >= ~100um of routing clearance) -- 550 covers
# the example sweep below (max coupling_length_um=400) with 150um to spare.
# Verified (rendered + checked for overlap) across that whole grid.
COUPLING_BUDGET_UM = 500.0
DOWN_LENGTH_UM = COUPLING_BUDGET_UM - COUPLING_LENGTH_UM

LP1, LP2, CLT, TL1, TL2 = build_feedline_and_coupler(
    COUPLING_SPACE_UM, COUPLING_LENGTH_UM, DOWN_LENGTH_UM)
design.rebuild()
gui.rebuild()
gui.autoscale()

In [5]:
L_PROBE_NH = 15
CJ_PROBE_FF = 1
L_TRANSMON_NH = 15
CJ_TRANSMON_FF = 1

In [ ]:
def build_squid_array(name, n_squids, pos_x_um, pos_y_um, orientation,
                      squid_gap_um, arm_width_um, josephson_gap_um,
                      segment_length_um, arm_semi_length_um, line_width_um,
                      L_j_nH, C_j_fF, jj_width_um, ground_gap_um=6.0):
    return SquidArray(design, name, options={
        'pos_x': f'{pos_x_um}um',
        'pos_y': f'{pos_y_um}um',
        'orientation': f'{orientation}',
        'n_squids': f'{n_squids}',
        'plate1_width': f'{arm_width_um}um',
        'squid_gap': f'{squid_gap_um}um',
        'segment_a_length': f'{arm_semi_length_um}um',
        'segment_a_width': f'{arm_width_um}um',
        'JJ_gap': f'{josephson_gap_um}um',
        'segment_b_length': f'{arm_semi_length_um}um',
        'segment_b_width': f'{arm_width_um}um',
        'segment_c_width': f'{arm_width_um}um',
        'segment_d_length': f'{segment_length_um}um',
        'segment_d_width': f'{line_width_um}um',
        'jj_width': f'{jj_width_um}um',
        'L_j': f'{L_j_nH}nH',
        'C_j': f'{C_j_fF}fF',
        'ground_gap': f'{ground_gap_um}um',
        'lead_plate1_width': '6um',
        'lead_segment_d_length': '6um',
        'lead_segment_d_width': '9um',
        'lead_plate2_width': '0um',
    })


try:
    array_1.delete()
except NameError:
    pass

array_1 = build_squid_array(
    name="array_1", n_squids=23, pos_x_um=-1290, pos_y_um=-400, orientation=0.0,
    squid_gap_um=25, arm_width_um=4, josephson_gap_um=3,
    segment_length_um=0, arm_semi_length_um=15, line_width_um=9,
    L_j_nH=L_PROBE_NH, C_j_fF=CJ_PROBE_FF, jj_width_um=2.0,
)

gui.rebuild()
gui.autoscale()

# 8 loops -> 16 junction rows, 9 galvanic nets; pins array_1:a (left) / array_1:b (right)

In [7]:
array_1.options

{'pos_x': '-1290um',
 'pos_y': '-400um',
 'orientation': '0.0',
 'chip': 'main',
 'layer': '1',
 'n_squids': '23',
 'plate1_width': '4um',
 'squid_gap': '25um',
 'segment_a_length': '15um',
 'segment_a_width': '4um',
 'segment_b_length': '15um',
 'segment_b_width': '4um',
 'JJ_gap': '3um',
 'segment_c_width': '4um',
 'segment_d_length': '0um',
 'segment_d_width': '9um',
 'plate2_width': '0um',
 'lead_plate1_width': '4um',
 'lead_segment_d_length': '6um',
 'lead_segment_d_width': '9um',
 'lead_plate2_width': '0um',
 'link_overlap': '0.5um',
 'jj_width': '2.0um',
 'L_j': '15nH',
 'C_j': '1fF',
 'ground_gap': '6.0um',
 'loop_gap': '0um',
 'hfss_inductance': '10nH',
 'hfss_capacitance': 0,
 'hfss_resistance': 0,
 'hfss_mesh_kw_jj': 7e-06,
 'q3d_inductance': '10nH',
 'q3d_capacitance': 0,
 'q3d_resistance': 0,
 'q3d_mesh_kw_jj': 7e-06,
 'gds_cell_name': 'my_other_junction',
 'aedt_q3d_inductance': 1e-08,
 'aedt_q3d_capacitance': 0,
 'aedt_hfss_inductance': 1e-08,
 'aedt_hfss_capacitance': 0

In [ ]:
try:
    qubit_1.delete()
except NameError : pass

qubit_options = Dict(
    pos_x='0um', pos_y='0um', orientation='0', chip='main',
    cpw_width='cpw_width', cpw_gap='cpw_gap',

    pad_radius='90um',
    pad_gap='10um',

    jj_options=Dict(
        jj_width='5um',
        jj_angle='0',
        L_j=f'{L_TRANSMON_NH}nH',
        C_j=f'{CJ_TRANSMON_FF}fF',
        export_mask=False,
        jj_sim_gap='3um',
    )
)
qubit_1 = CircTransmon(design, "Q1", options=qubit_options)

gui.rebuild()
gui.autoscale()

In [14]:
try: 
    cutout.delete()
except NameError : pass

cutout_options = {
    'pos_x': '-750um',
    'pos_y': '-400um',
    'orientation': '0',
    'chip': 'main',
    'layer': '1',
    'width': '1100um',
    'height': '80um'
    }

cutout = RectCutout(design, "cutout_1", options = cutout_options)

gui.rebuild()
gui.autoscale()

In [ ]:
# cutout.delete()
# gui.rebuild()
